# Generate Pipeline (Refined)

요청사항을 반영한 `generate` 단계 정리 노트북입니다.

## 반영 사항
1. 각 recipe에서 `S(r,m)` 상위 200개 molecule만 유지
2. graph 산출물은 cuisine별 폴더(`result/graph/<cuisine_key>/`)로 저장
3. 현재는 `Thai`, `Korean`만 실행(변수로 쉽게 확장 가능)
4. recipe→molecule edge csv에 `name/rank/molecule/molecule_name/flavor_profile/p/S_rm` 포함
5. molecule weight csv에도 `molecule_name/flavor_profile` 포함
6. molecule 관점(top recipes per molecule) csv 추가

## 핵심 수식
\[
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdotrac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
\]

그리고 각 recipe 내부에서 `p = S(r,m) / \sum_m S(r,m)`로 정규화합니다.

## 0) 환경 설정
- 긴 반복은 `tqdm`으로 진행률 표시
- 중간 산출물 `shape`, `head(10)` 확인

In [ ]:
from __future__ import annotations

import ast
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

START_TS = time.time()

In [ ]:
# Paths
RECIPES_PATH = Path('../recipes.csv')
FLAVORDB_PATH = Path('../preprocess/flavordb_alias_in_vocab_only.csv')
MOLECULES_PATH = Path('../molecules.csv')

OUT_DIR = Path('../result')
GRAPH_DIR = OUT_DIR / 'graph'
for d in [OUT_DIR, GRAPH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Controls
TARGET_CUISINES = ['Thai', 'Korean']  # 나중에 None 또는 []로 바꿔 전체 cuisine 처리 가능
TOP_K_PER_RECIPE = 200
TOP_N_RECIPES_PER_MOLECULE = 200

print('TARGET_CUISINES =', TARGET_CUISINES)
print('TOP_K_PER_RECIPE =', TOP_K_PER_RECIPE)
print('TOP_N_RECIPES_PER_MOLECULE =', TOP_N_RECIPES_PER_MOLECULE)

## 1) 데이터 로드 + 유틸

In [ ]:
def norm_text(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r'[_\-]+', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r"[^\w\s']", '', s)
    return s.strip()


def safe_key(name: str) -> str:
    return re.sub(r'\W+', '_', str(name)).strip('_') or 'unknown'


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cmap = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cmap:
            return cmap[cand.lower()]
    return None


def parse_ingredient_ratio(raw):
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        txt = raw.strip()
        if not txt:
            return []
        try:
            raw = ast.literal_eval(txt)
        except Exception:
            return []

    out = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            try:
                out.append((norm_text(k), float(v)))
            except Exception:
                continue
    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                ing = item.get('ingredient', item.get('name', item.get('ing')))
                val = item.get('ratio', item.get('weight', item.get('w', item.get('gram', 0))))
                try:
                    out.append((norm_text(ing), float(val)))
                except Exception:
                    continue
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                try:
                    out.append((norm_text(item[0]), float(item[1])))
                except Exception:
                    continue
    return [(i, w) for i, w in out if i and np.isfinite(w) and w > 0]

In [ ]:
recipes = pd.read_csv(RECIPES_PATH)
flavordb = pd.read_csv(FLAVORDB_PATH)
molecules = pd.read_csv(MOLECULES_PATH)

print('recipes:', recipes.shape)
print('flavordb:', flavordb.shape)
print('molecules:', molecules.shape)

display(recipes.head(10))
display(flavordb.head(10))
display(molecules.head(10))

In [ ]:
recipe_id_col = find_col(recipes, ['recipe_id', 'id', 'rid'])
recipe_name_col = find_col(recipes, ['name', 'recipe_name', 'title'])
cuisine_col = find_col(recipes, ['cuisine', 'cuisine_name'])
ratio_col = find_col(recipes, ['ingredients_ratio', 'ingredient_ratio', 'ingredients_with_ratio'])
if ratio_col is None:
    ratio_col = find_col(recipes, ['cleaned_ingredients', 'ingredients'])

if cuisine_col is None or ratio_col is None:
    raise KeyError(f'Missing required columns: cuisine={cuisine_col}, ratio={ratio_col}, all={list(recipes.columns)}')

if recipe_id_col is None:
    recipes = recipes.copy()
    recipes['recipe_id'] = np.arange(len(recipes), dtype=int)
    recipe_id_col = 'recipe_id'

if recipe_name_col is None:
    recipes = recipes.copy()
    recipes['name'] = recipes[recipe_id_col].astype(str)
    recipe_name_col = 'name'

ing_col = find_col(flavordb, ['ingredient', 'entity_alias_readable', 'entity_alias', 'name'])
mol_col = find_col(flavordb, ['molecule id', 'molecule_id', 'pubchem id', 'pubchem_id', 'molecule'])

molecules_id_col = find_col(molecules, ['molecule', 'molecule_id', 'pubchem id', 'pubchem_id'])
molecules_name_col = find_col(molecules, ['molecule_name', 'name'])
molecules_fp_col = find_col(molecules, ['flavor_profile', 'flavor profile'])

if ing_col is None or mol_col is None:
    raise KeyError(f'Missing flavordb columns: ingredient={ing_col}, molecule={mol_col}')
if molecules_id_col is None:
    raise KeyError(f'Missing molecule id column in molecules.csv: {list(molecules.columns)}')

print('recipe_id_col =', recipe_id_col)
print('recipe_name_col =', recipe_name_col)
print('cuisine_col =', cuisine_col)
print('ratio_col =', ratio_col)
print('ing_col =', ing_col)
print('mol_col =', mol_col)
print('molecules_id_col =', molecules_id_col)
print('molecules_name_col =', molecules_name_col)
print('molecules_fp_col =', molecules_fp_col)

## 2) ingredient→molecule 매핑 및 molecule 메타

In [ ]:
flavordb = flavordb.copy()
flavordb['ingredient_norm'] = flavordb[ing_col].map(norm_text)
flavordb['molecule_id'] = pd.to_numeric(flavordb[mol_col], errors='coerce')

ing_to_molecules = (
    flavordb.dropna(subset=['ingredient_norm', 'molecule_id'])
    .groupby('ingredient_norm')['molecule_id']
    .apply(lambda x: sorted(set(int(v) for v in x if pd.notna(v))))
    .to_dict()
)

mol_meta = molecules.copy()
mol_meta['molecule_id'] = pd.to_numeric(mol_meta[molecules_id_col], errors='coerce')
mol_meta = mol_meta.dropna(subset=['molecule_id']).copy()
mol_meta['molecule_id'] = mol_meta['molecule_id'].astype(int)

if molecules_name_col is not None:
    mol_meta['molecule_name'] = mol_meta[molecules_name_col].astype(str)
else:
    mol_meta['molecule_name'] = '(unknown)'

if molecules_fp_col is not None:
    mol_meta['flavor_profile'] = mol_meta[molecules_fp_col].astype(str)
else:
    mol_meta['flavor_profile'] = ''

mol_meta = mol_meta[['molecule_id', 'molecule_name', 'flavor_profile']].drop_duplicates('molecule_id')
id2name = dict(zip(mol_meta['molecule_id'], mol_meta['molecule_name']))
id2fp = dict(zip(mol_meta['molecule_id'], mol_meta['flavor_profile']))

print('mapped ingredients:', len(ing_to_molecules))
print('molecule meta size:', mol_meta.shape)
display(mol_meta.head(10))

## 3) recipes long 생성 + cuisine 필터(Thai/Korean) + FlavorDB 매칭 필터

In [ ]:
records = []
for _, row in tqdm(recipes.iterrows(), total=len(recipes), desc='Parse recipes'):
    rid = int(row[recipe_id_col])
    rname = str(row[recipe_name_col])
    cuisine = str(row[cuisine_col])
    pairs = parse_ingredient_ratio(row[ratio_col])
    z = sum(w for _, w in pairs)
    if z <= 0:
        continue
    for ing, w in pairs:
        records.append({
            'recipe_id': rid,
            'name': rname,
            'cuisine': cuisine,
            'ingredient': ing,
            'w_ri': float(w / z),
        })

recipes_long = pd.DataFrame(records)
print('recipes_long:', recipes_long.shape)
display(recipes_long.head(10))

if TARGET_CUISINES:
    recipes_long = recipes_long[recipes_long['cuisine'].isin(TARGET_CUISINES)].copy()
    print('after cuisine filter:', recipes_long.shape)

recipes_long['has_flavordb_match'] = recipes_long['ingredient'].isin(ing_to_molecules)
matched_rids = set(recipes_long.loc[recipes_long['has_flavordb_match'], 'recipe_id'].unique())
recipes_long = recipes_long[recipes_long['recipe_id'].isin(matched_rids)].copy()

print('after flavordb-recipe filter:', recipes_long.shape)
print('n_recipes:', recipes_long['recipe_id'].nunique())
display(recipes_long.head(10))

## 4) S(r,m) 계산 → recipe별 상위 200개 → p 정규화

In [ ]:
score_rows = []
for rid, grp in tqdm(recipes_long.groupby('recipe_id'), desc='Compute S(r,m)'):
    acc = defaultdict(float)
    for ing, w in grp[['ingredient', 'w_ri']].itertuples(index=False):
        mols = ing_to_molecules.get(ing, [])
        if not mols:
            continue
        contrib = w / len(mols)
        for m in mols:
            acc[m] += contrib

    if not acc:
        continue

    sorted_items = sorted(acc.items(), key=lambda x: x[1], reverse=True)[:TOP_K_PER_RECIPE]
    denom = sum(v for _, v in sorted_items)
    if denom <= 0:
        continue

    row0 = grp.iloc[0]
    for rank, (mid, srm) in enumerate(sorted_items, start=1):
        p = srm / denom
        score_rows.append({
            'recipe_id': int(rid),
            'name': row0['name'],
            'cuisine': row0['cuisine'],
            'rank': rank,
            'molecule': int(mid),
            'p': float(p),
            'S_rm': float(srm),
        })

recipe_molecule_top = pd.DataFrame(score_rows)
recipe_molecule_top['molecule_name'] = recipe_molecule_top['molecule'].map(id2name).fillna('(unknown)')
recipe_molecule_top['flavor_profile'] = recipe_molecule_top['molecule'].map(id2fp).fillna('')

print('recipe_molecule_top:', recipe_molecule_top.shape)
display(recipe_molecule_top.head(10))

check = recipe_molecule_top.groupby('recipe_id', as_index=False)['p'].sum().rename(columns={'p':'sum_p'})
print('p-sum stats by recipe:')
print(check['sum_p'].describe())
display(check.head(10))

## 5) cuisine별 산출물 저장 (폴더 구조 포함)

In [ ]:
def save_cuisine_outputs(cuisine_key: str, df: pd.DataFrame):
    out_dir = GRAPH_DIR / cuisine_key
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) recipe-molecule edge view
    edges = df[['recipe_id', 'name', 'rank', 'molecule', 'molecule_name', 'flavor_profile', 'p', 'S_rm']].copy()
    edges = edges.sort_values(['recipe_id', 'rank'])
    edges_path = out_dir / '000_recipe_molecule_edges.csv'
    edges.to_csv(edges_path, index=False)

    # 2) molecule weight view
    mol_weight = (
        df.groupby('molecule', as_index=False)
        .agg(weight=('p', 'sum'), recipe_count=('recipe_id', 'nunique'))
        .sort_values('weight', ascending=False)
    )
    mol_weight['molecule_name'] = mol_weight['molecule'].map(id2name).fillna('(unknown)')
    mol_weight['flavor_profile'] = mol_weight['molecule'].map(id2fp).fillna('')
    mol_weight = mol_weight[['molecule', 'molecule_name', 'flavor_profile', 'weight', 'recipe_count']]
    mol_weight_path = out_dir / '001_molecule_weight.csv'
    mol_weight.to_csv(mol_weight_path, index=False)

    # 3) molecule -> top recipes view
    reverse_rows = []
    for mid, g in df.groupby('molecule'):
        g = g.sort_values('p', ascending=False).head(TOP_N_RECIPES_PER_MOLECULE)
        for rnk, row in enumerate(g.itertuples(index=False), start=1):
            reverse_rows.append({
                'molecule': int(mid),
                'molecule_name': getattr(row, 'molecule_name'),
                'flavor_profile': getattr(row, 'flavor_profile'),
                'rank': rnk,
                'recipe_id': int(getattr(row, 'recipe_id')),
                'name': getattr(row, 'name'),
                'p': float(getattr(row, 'p')),
                'S_rm': float(getattr(row, 'S_rm')),
            })

    reverse_df = pd.DataFrame(reverse_rows)
    reverse_path = out_dir / '002_molecule_top_recipes.csv'
    reverse_df.to_csv(reverse_path, index=False)

    print(f'[{cuisine_key}] edges={edges.shape}, molecule_weight={mol_weight.shape}, reverse={reverse_df.shape}')
    display(edges.head(10))
    display(mol_weight.head(10))
    display(reverse_df.head(10))


selected_cuisines = sorted(recipe_molecule_top['cuisine'].dropna().unique().tolist())
for cuisine in tqdm(selected_cuisines, desc='Save by cuisine'):
    ckey = safe_key(cuisine)
    sub = recipe_molecule_top[recipe_molecule_top['cuisine'] == cuisine].copy()
    save_cuisine_outputs(ckey, sub)

# optional ALL (현재는 설정단계라 기본 off, 필요시 True로)
EXPORT_ALL = False
if EXPORT_ALL:
    save_cuisine_outputs('ALL', recipe_molecule_top.copy())

## 6) 요약

In [ ]:
summary = (
    recipe_molecule_top.groupby('cuisine', as_index=False)
    .agg(n_recipes=('recipe_id', 'nunique'), n_rows=('recipe_id', 'size'), n_molecules=('molecule', 'nunique'))
    .sort_values('cuisine')
)

print('summary:', summary.shape)
display(summary.head(10))

summary.to_csv(OUT_DIR / 'generate_summary.csv', index=False)

elapsed = time.time() - START_TS
print(f'Done. elapsed={elapsed:.1f}s')
print('Graph output root:', GRAPH_DIR.resolve())